In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/playground-series-s6e4/sample_submission.csv
/kaggle/input/competitions/playground-series-s6e4/train.csv
/kaggle/input/competitions/playground-series-s6e4/test.csv


In [3]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

In [4]:
train = pd.read_csv("/kaggle/input/competitions/playground-series-s6e4/train.csv")
test = pd.read_csv("/kaggle/input/competitions/playground-series-s6e4/test.csv")
sample = pd.read_csv("/kaggle/input/competitions/playground-series-s6e4/sample_submission.csv")

In [40]:
test.head()

,id,Soil_Type,Soil_pH,Soil_Moisture,Organic_Carbon,Electrical_Conductivity,Temperature_C,Humidity,Rainfall_mm,Sunlight_Hours,Wind_Speed_kmh,Crop_Type,Crop_Growth_Stage,Season,Irrigation_Type,Water_Source,Field_Area_hectare,Mulching_Used,Previous_Irrigation_mm,Region
0,630000,Silt,6.36,26.19,0.59,2.81,17.83,30.24,1533.38,5.40,3.00,Maize,Sowing,Rabi,Canal,River,13.59,Yes,47.48,West
1,630001,Clay,5.87,9.88,1.18,3.26,21.18,78.07,576.05,7.22,15.88,Cotton,Sowing,Rabi,Drip,Reservoir,6.12,Yes,56.43,South
2,630002,Sandy,6.22,26.55,0.96,0.85,26.87,60.35,545.30,9.43,2.63,Wheat,Sowing,Kharif,Sprinkler,Reservoir,3.11,Yes,20.00,East
3,630003,Clay,7.68,53.58,0.83,0.55,41.74,36.05,1211.03,6.69,1.86,Maize,Harvest,Rabi,Canal,Groundwater,2.27,No,102.99,North
4,630004,Loamy,5.23,59.02,0.54,2.11,41.08,52.47,1321.91,4.11,5.71,Cotton,Sowing,Kharif,Canal,Groundwater,12.39,Yes,13.33,Central


In [28]:
train.head(20)

,id,Soil_Type,Soil_pH,Soil_Moisture,Organic_Carbon,Electrical_Conductivity,Temperature_C,Humidity,Rainfall_mm,Sunlight_Hours,...,Crop_Type,Crop_Growth_Stage,Season,Irrigation_Type,Water_Source,Field_Area_hectare,Mulching_Used,Previous_Irrigation_mm,Region,Irrigation_Need
0,0,Loamy,4.92,32.58,1.01,3.05,15.01,50.61,725.99,5.90,...,Sugarcane,Sowing,Zaid,Drip,Rainwater,0.82,No,112.16,East,0
1,1,Clay,7.08,56.61,0.44,2.00,22.92,67.86,985.66,6.98,...,Wheat,Vegetative,Kharif,Rainfed,River,5.27,Yes,47.16,South,0
2,2,Clay,5.69,27.71,0.81,2.83,26.97,92.22,2201.70,6.05,...,Rice,Vegetative,Kharif,Sprinkler,Reservoir,8.24,Yes,110.38,North,0
3,3,Sandy,5.65,13.32,1.33,0.87,13.32,61.57,1357.33,9.12,...,Wheat,Flowering,Kharif,Canal,River,8.32,Yes,53.85,South,1
4,4,Clay,7.96,59.14,0.38,0.96,20.22,91.11,1538.20,6.95,...,Wheat,Sowing,Rabi,Canal,River,7.37,No,93.19,South,0
5,5,Sandy,5.09,24.70,1.28,0.48,12.21,92.35,696.03,9.11,...,Sugarcane,Flowering,Kharif,Sprinkler,River,0.81,No,86.56,South,1
6,6,Silt,7.53,49.67,1.44,1.62,14.02,61.65,889.39,8.44,...,Potato,Flowering,Zaid,Sprinkler,Rainwater,13.32,No,14.05,East,1
7,7,Loamy,7.56,48.61,0.38,1.31,22.78,61.53,708.82,9.96,...,Rice,Sowing,Zaid,Sprinkler,Reservoir,5.63,Yes,110.99,East,0
8,8,Silt,6.02,53.01,0.90,0.49,20.55,61.30,1536.36,10.50,...,Potato,Flowering,Kharif,Rainfed,River,8.17,Yes,36.37,West,0
9,9,Silt,7.39,41.91,0.58,0.78,39.25,85.52,281.76,8.86,...,Wheat,Vegetative,Kharif,Rainfed,Reservoir,5.41,No,71.92,Central,2


In [6]:
print(train['Irrigation_Need'].value_counts(normalize=True) * 100)

Irrigation_Need
Low       58.716984
Medium    37.948254
High       3.334762
Name: proportion, dtype: float64


In [7]:
train["Irrigation_Need"] = train["Irrigation_Need"].map({'Low':0, 'Medium':1, 'High':2})

In [21]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 630000 entries, 0 to 629999
Data columns (total 21 columns):
 #   Column                   Non-Null Count   Dtype  
---  ------                   --------------   -----  
 0   id                       630000 non-null  int64  
 1   Soil_Type                630000 non-null  object 
 2   Soil_pH                  630000 non-null  float64
 3   Soil_Moisture            630000 non-null  float64
 4   Organic_Carbon           630000 non-null  float64
 5   Electrical_Conductivity  630000 non-null  float64
 6   Temperature_C            630000 non-null  float64
 7   Humidity                 630000 non-null  float64
 8   Rainfall_mm              630000 non-null  float64
 9   Sunlight_Hours           630000 non-null  float64
 10  Wind_Speed_kmh           630000 non-null  float64
 11  Crop_Type                630000 non-null  object 
 12  Crop_Growth_Stage        630000 non-null  object 
 13  Season                   630000 non-null  object 
 14  Irri

In [35]:
train.head(10).T

,0,1,2,3,4,5,6,7,8,9
id,0,1,2,3,4,5,6,7,8,9
Soil_Type,Loamy,Clay,Clay,Sandy,Clay,Sandy,Silt,Loamy,Silt,Silt
Soil_pH,4.92,7.08,5.69,5.65,7.96,5.09,7.53,7.56,6.02,7.39
Soil_Moisture,32.58,56.61,27.71,13.32,59.14,24.7,49.67,48.61,53.01,41.91
Organic_Carbon,1.01,0.44,0.81,1.33,0.38,1.28,1.44,0.38,0.9,0.58
Electrical_Conductivity,3.05,2.0,2.83,0.87,0.96,0.48,1.62,1.31,0.49,0.78
Temperature_C,15.01,22.92,26.97,13.32,20.22,12.21,14.02,22.78,20.55,39.25
Humidity,50.61,67.86,92.22,61.57,91.11,92.35,61.65,61.53,61.3,85.52
Rainfall_mm,725.99,985.66,2201.7,1357.33,1538.2,696.03,889.39,708.82,1536.36,281.76
Sunlight_Hours,5.9,6.98,6.05,9.12,6.95,9.11,8.44,9.96,10.5,8.86


In [48]:
cat_cols = ["Soil_Type", "Crop_Type","Crop_Growth_Stage", "Season", "Irrigation_Type", "Water_Source","Mulching_Used","Region"]

In [14]:
test_ids = test['id'] 
train_ids = train['id']
test = test.drop(columns=['id']) 
train= train.drop(columns=['id'])

In [59]:
test.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 270000 entries, 0 to 269999
Data columns (total 23 columns):
 #   Column                   Non-Null Count   Dtype   
---  ------                   --------------   -----   
 0   id                       270000 non-null  int64   
 1   Soil_Type                270000 non-null  category
 2   Soil_pH                  270000 non-null  float64 
 3   Soil_Moisture            270000 non-null  float64 
 4   Organic_Carbon           270000 non-null  float64 
 5   Electrical_Conductivity  270000 non-null  float64 
 6   Temperature_C            270000 non-null  float64 
 7   Humidity                 270000 non-null  float64 
 8   Rainfall_mm              270000 non-null  float64 
 9   Sunlight_Hours           270000 non-null  float64 
 10  Wind_Speed_kmh           270000 non-null  float64 
 11  Crop_Type                270000 non-null  category
 12  Crop_Growth_Stage        270000 non-null  category
 13  Season                   270000 non-null  ca

In [60]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 630000 entries, 0 to 629999
Data columns (total 24 columns):
 #   Column                   Non-Null Count   Dtype   
---  ------                   --------------   -----   
 0   id                       630000 non-null  int64   
 1   Soil_Type                630000 non-null  category
 2   Soil_pH                  630000 non-null  float64 
 3   Soil_Moisture            630000 non-null  float64 
 4   Organic_Carbon           630000 non-null  float64 
 5   Electrical_Conductivity  630000 non-null  float64 
 6   Temperature_C            630000 non-null  float64 
 7   Humidity                 630000 non-null  float64 
 8   Rainfall_mm              630000 non-null  float64 
 9   Sunlight_Hours           630000 non-null  float64 
 10  Wind_Speed_kmh           630000 non-null  float64 
 11  Crop_Type                630000 non-null  category
 12  Crop_Growth_Stage        630000 non-null  category
 13  Season                   630000 non-null  ca

In [64]:
print("Train columns:", X_train.columns)
print("Test columns:", test.columns)

Train columns: Index(['Soil_Type', 'Soil_pH', 'Soil_Moisture', 'Organic_Carbon',
       'Electrical_Conductivity', 'Temperature_C', 'Humidity', 'Rainfall_mm',
       'Sunlight_Hours', 'Wind_Speed_kmh', 'Crop_Type', 'Crop_Growth_Stage',
       'Season', 'Irrigation_Type', 'Water_Source', 'Field_Area_hectare',
       'Mulching_Used', 'Previous_Irrigation_mm', 'Region', 'Weather_Stress',
       'Water_Need', 'Crop_Soil'],
      dtype='object')
Test columns: Index(['id', 'Soil_Type', 'Soil_pH', 'Soil_Moisture', 'Organic_Carbon',
       'Electrical_Conductivity', 'Temperature_C', 'Humidity', 'Rainfall_mm',
       'Sunlight_Hours', 'Wind_Speed_kmh', 'Crop_Type', 'Crop_Growth_Stage',
       'Season', 'Irrigation_Type', 'Water_Source', 'Field_Area_hectare',
       'Mulching_Used', 'Previous_Irrigation_mm', 'Region', 'Weather_Stress',
       'Water_Need', 'Crop_Soil'],
      dtype='object')


In [65]:
print(set(test.columns) - set(X_train.columns))   # extra in test
print(set(X_train.columns) - set(test.columns))   # missing in test

{'id'}
set()


In [23]:
train.head(15).T

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14
Soil_Type,Loamy,Clay,Clay,Sandy,Clay,Sandy,Silt,Loamy,Silt,Silt,Loamy,Loamy,Silt,Sandy,Sandy
Soil_pH,4.92,7.08,5.69,5.65,7.96,5.09,7.53,7.56,6.02,7.39,5.44,5.95,7.56,5.37,6.65
Soil_Moisture,32.58,56.61,27.71,13.32,59.14,24.7,49.67,48.61,53.01,41.91,52.99,57.67,11.12,56.11,52.42
Organic_Carbon,1.01,0.44,0.81,1.33,0.38,1.28,1.44,0.38,0.9,0.58,1.16,0.63,1.56,0.33,0.32
Electrical_Conductivity,3.05,2.0,2.83,0.87,0.96,0.48,1.62,1.31,0.49,0.78,3.0,2.19,3.32,1.37,0.82
Temperature_C,15.01,22.92,26.97,13.32,20.22,12.21,14.02,22.78,20.55,39.25,25.03,17.42,32.71,23.65,39.62
Humidity,50.61,67.86,92.22,61.57,91.11,92.35,61.65,61.53,61.3,85.52,75.67,51.67,57.94,43.75,86.63
Rainfall_mm,725.99,985.66,2201.7,1357.33,1538.2,696.03,889.39,708.82,1536.36,281.76,1419.68,1642.63,719.5,1313.04,2060.59
Sunlight_Hours,5.9,6.98,6.05,9.12,6.95,9.11,8.44,9.96,10.5,8.86,9.31,5.64,7.93,4.72,9.09
Wind_Speed_kmh,16.79,3.39,3.85,2.31,13.94,7.37,16.76,7.83,19.38,13.13,4.79,14.85,13.85,10.57,6.07


In [33]:
cat_cols = [
    'Soil_Type', 'Crop_Type', 'Crop_Growth_Stage',
    'Season', 'Irrigation_Type', 'Water_Source',
    'Mulching_Used', 'Region', 'Crop_Soil'
]

for col in cat_cols:
    train[col] = train[col].astype('category')
    test[col] = test[col].astype('category')

In [34]:
X = train.drop(columns=['Irrigation_Need'])
y = train['Irrigation_Need']

In [35]:
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(X,y, test_size=0.2, random_state=42, stratify=y)

In [19]:
print(f"Traning on {X_train.shape[0]}rows.")

print(f"Validating on{X_train.shape[0]}rows.")

Traning on 504000rows.
Validating on504000rows.


In [79]:
print(f"Check: {X_train.shape[0]} + {X_val.shape[0]} = {X_train.shape[0] + X_val.shape[0]}")

Check: 504000 + 126000 = 630000


In [8]:
train['Weather_Stress'] = (
    train['Temperature_C'] * train['Wind_Speed_kmh']
) / (train['Humidity'] + 1)

In [9]:
test['Weather_Stress'] = (
    test['Temperature_C'] * test['Wind_Speed_kmh']
) / (test['Humidity'] + 1)

In [10]:
train['Water_Need'] = (
    train['Temperature_C'] * train['Sunlight_Hours']
) / (train['Rainfall_mm'] + 1)

In [11]:
test['Water_Need'] = (
    test['Temperature_C'] * test['Sunlight_Hours']
) / (test['Rainfall_mm'] + 1)

In [12]:
train['Crop_Soil'] = train['Crop_Type'] + "_" + train['Soil_Type']

In [13]:
test['Crop_Soil'] = test['Crop_Type'] + "_" + test['Soil_Type']

In [31]:
cat_cols = X.select_dtypes(include=['object']).columns.tolist()

print(cat_cols)

[]


In [66]:
test_ids = test['id']

In [41]:
test = test.drop(['id'], axis=1)

KeyError: "['id'] not found in axis"

In [36]:
from catboost import CatBoostClassifier
from sklearn.model_selection import train_test_split


X_train, X_val, y_train, y_val = train_test_split(X,y, test_size=0.2, random_state=42, stratify=y)


model = CatBoostClassifier(
    iterations=10000,
    learning_rate=0.001,
    l2_leaf_reg=5,
    depth=6,
    task_type='GPU',
    devices='0',
    early_stopping_rounds=100,
    verbose=500
)

model.fit(
    X_train, y_train,
    eval_set=(X_val, y_val),
    cat_features=cat_cols
)

0:	learn: 1.0967118	test: 1.0967220	best: 1.0967220 (0)	total: 7.04s	remaining: 19h 34m 11s
500:	learn: 0.5472530	test: 0.5474184	best: 0.5474184 (500)	total: 14.6s	remaining: 4m 37s
1000:	learn: 0.3240702	test: 0.3242911	best: 0.3242911 (1000)	total: 22.3s	remaining: 3m 20s
1500:	learn: 0.2119225	test: 0.2121729	best: 0.2121729 (1500)	total: 30s	remaining: 2m 49s
2000:	learn: 0.1513945	test: 0.1516569	best: 0.1516569 (2000)	total: 37.8s	remaining: 2m 31s
2500:	learn: 0.1173340	test: 0.1176080	best: 0.1176080 (2500)	total: 45.5s	remaining: 2m 16s
3000:	learn: 0.0976075	test: 0.0979316	best: 0.0979316 (3000)	total: 52.9s	remaining: 2m 3s
3500:	learn: 0.0857816	test: 0.0861643	best: 0.0861643 (3500)	total: 60s	remaining: 1m 51s
4000:	learn: 0.0786316	test: 0.0790731	best: 0.0790731 (4000)	total: 1m 7s	remaining: 1m 40s
4500:	learn: 0.0743291	test: 0.0748221	best: 0.0748221 (4500)	total: 1m 14s	remaining: 1m 30s
5000:	learn: 0.0716613	test: 0.0722034	best: 0.0722034 (5000)	total: 1m 21s	r

CatBoostClassifier(depth=6, devices='0', early_stopping_rounds=100, iterations=10000, l2_leaf_reg=5, learning_rate=0.001, task_type='GPU', verbose=500)

In [37]:

test = test[X.columns]

from catboost import Pool
test_pool = Pool(test, cat_features=cat_cols)


test_preds = model.predict_proba(test_pool)

In [40]:
reverse_map = {
    0: "Low",
    1: "Medium",
    2: "High"
}

In [41]:
test_preds = model.predict(test_pool)
test_preds = test_preds.flatten()

# Convert numbers → labels
test_preds = pd.Series(test_preds).map(reverse_map)

In [43]:
submission = pd.DataFrame({
    'id': test_ids,
    'Irrigation_Need': test_preds
})

submission.to_csv("submission1.csv", index=False)

In [20]:
from sklearn.preprocessing import LabelEncoder

cat_cols = train.select_dtypes(include='object').columns

for col in cat_cols:
    combined = pd.concat([train[col], test[col]]).astype(str)
    le = LabelEncoder()
    le.fit(combined)

    train[col] = le.transform(train[col].astype(str))
    test[col] = le.transform(test[col].astype(str))

In [21]:
from lightgbm import LGBMClassifier

model = LGBMClassifier(
    device='gpu',          
    gpu_platform_id=0,
    gpu_device_id=0,
    n_estimators=5000,
    learning_rate=0.001,
    max_depth=6,
    num_leaves=100,
    subsample=0.8,
    colsample_bytree=0.8,
)

In [23]:
from lightgbm import early_stopping, log_evaluation

model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    eval_metric='multi_logloss',
    callbacks=[
        early_stopping(100),
        log_evaluation(100)]
)

[LightGBM] [Info] This is the GPU trainer!!
[LightGBM] [Info] Total Bins 3237
[LightGBM] [Info] Number of data points in the train set: 504000, number of used features: 22
[LightGBM] [Info] Using requested OpenCL platform 0 device 0
[LightGBM] [Info] Using GPU Device: Tesla P100-PCIE-16GB, Vendor: NVIDIA Corporation
[LightGBM] [Info] Compiling OpenCL Kernel with 256 bins...
[LightGBM] [Info] GPU programs have been built
[LightGBM] [Info] Size of histogram bin entry: 8
[LightGBM] [Info] 22 dense feature groups (11.54 MB) transferred to GPU in 0.012026 secs. 0 sparse feature groups
[LightGBM] [Info] Start training from score -0.532440
[LightGBM] [Info] Start training from score -0.968948
[LightGBM] [Info] Start training from score -3.400781
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
Training until validation 

Exception ignored on calling ctypes callback function: <function _log_callback at 0x7c94f09a6b60>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/lightgbm/basic.py", line 287, in _log_callback
    def _log_callback(msg: bytes) -> None:
    
KeyboardInterrupt: 


[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with po

LGBMClassifier(colsample_bytree=0.8, device='gpu', gpu_device_id=0,
               gpu_platform_id=0, learning_rate=0.001, max_depth=6,
               n_estimators=5000, num_leaves=100, subsample=0.8)

In [24]:
preds = model.predict(test)

reverse_map = {0: "Low", 1: "Medium", 2: "High"}
preds = pd.Series(preds).map(reverse_map)


submission = pd.DataFrame({
    'id': test_ids,
    'Irrigation_Need': preds
})


submission.to_csv("submission.csv", index=False)